In [6]:
# Cell 1: install (run once, takes ~30 seconds)
!pip install timm --quiet


In [7]:
# Cell 2 (revised): load checkpoint directly, build matching architecture
import torch
import torch.nn as nn
from huggingface_hub import hf_hub_download

# Download the checkpoint
ckpt_path = hf_hub_download(repo_id="edadaltocg/resnet50_simclr_cifar10", filename="pytorch_model.bin")
state_dict = torch.load(ckpt_path, map_location="cpu")

# Print the first 10 keys so we can see what's in it
for k in list(state_dict.keys())[:10]:
    print(k)

conv1.weight
bn1.weight
bn1.bias
bn1.running_mean
bn1.running_var
bn1.num_batches_tracked
layer1.0.conv1.weight
layer1.0.bn1.weight
layer1.0.bn1.bias
layer1.0.bn1.running_mean


In [8]:
import torch
import torch.nn as nn
import torchvision.models as tvm

# Build the CIFAR ResNet-50
base = tvm.resnet50(weights=None)
base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
base.maxpool = nn.Identity()

# Load weights into base BEFORE wrapping (keys match here)
result = base.load_state_dict(state_dict, strict=False)
print("Missing:", result.missing_keys)
print("Unexpected:", result.unexpected_keys)

# Now drop the fc head and wrap
base.fc = nn.Identity()
base.eval()

dummy = torch.randn(1, 3, 32, 32)
with torch.no_grad():
    out = base(dummy)
print("Feature shape:", out.shape)
print("Done.")

Missing: ['fc.weight', 'fc.bias']
Unexpected: []
Feature shape: torch.Size([1, 2048])
Done.


In [9]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 5: get CIFAR-10 into the folder torchvision expects.
# torchvision's CIFAR10(root="/tmp/cifar10") looks for /tmp/cifar10/cifar-10-batches-py.
# We extract the tarball there directly so download=True finds it and skips the slow
# Toronto re-download that stalled earlier (the bug was a path mismatch, not the mirror).
import os, subprocess

CIFAR_ROOT = "/tmp/cifar10"
TARBALL = "/tmp/cifar10.tar.gz"
os.makedirs(CIFAR_ROOT, exist_ok=True)

if os.path.isdir(os.path.join(CIFAR_ROOT, "cifar-10-batches-py")):
    print("CIFAR-10 already extracted at", CIFAR_ROOT)
else:
    if not os.path.exists(TARBALL):
        subprocess.run([
            "wget", "-q", "--show-progress",
            "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz",
            "-O", TARBALL,
        ], check=True)
    subprocess.run(["tar", "-xzf", TARBALL, "-C", CIFAR_ROOT], check=True)
    print("Extracted to", os.path.join(CIFAR_ROOT, "cifar-10-batches-py"))


In [ ]:
# Cell 6: extract frozen features (with Drive cache guard).
# Reruns are instant: if features.pt/labels.pt already exist on Drive, just load them.
import os
import torch
import torchvision
import torchvision.transforms as transforms

SAVE_DIR = "/content/drive/MyDrive/cifar10_simclr"
os.makedirs(SAVE_DIR, exist_ok=True)
FEAT_PATH = os.path.join(SAVE_DIR, "features.pt")
LBL_PATH  = os.path.join(SAVE_DIR, "labels.pt")

if os.path.exists(FEAT_PATH) and os.path.exists(LBL_PATH):
    feats_matrix = torch.load(FEAT_PATH)
    labels_vec   = torch.load(LBL_PATH)
    print("Loaded cached features from", SAVE_DIR)
    print("Features shape:", feats_matrix.shape, " Labels shape:", labels_vec.shape)
else:
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])
    # download=True is safe now: the previous cell placed the data at /tmp/cifar10,
    # so the integrity check passes and nothing is re-downloaded.
    dataset = torchvision.datasets.CIFAR10(root="/tmp/cifar10", train=True,
                                            download=True, transform=transform)
    loader = torch.utils.data.DataLoader(dataset, batch_size=256,
                                         shuffle=False, num_workers=2)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)
    base.to(device)
    base.eval()

    all_feats, all_labels = [], []
    with torch.no_grad():
        for i, (imgs, labels) in enumerate(loader):
            imgs = imgs.to(device)
            feats = base(imgs)                               # (B, 2048)
            feats = feats / feats.norm(dim=1, keepdim=True)  # L2 normalize
            all_feats.append(feats.cpu())
            all_labels.append(labels)
            if i % 10 == 0:
                print(f"Batch {i}/{len(loader)}")

    feats_matrix = torch.cat(all_feats, dim=0)   # (50000, 2048)
    labels_vec   = torch.cat(all_labels, dim=0)  # (50000,)

    torch.save(feats_matrix, FEAT_PATH)
    torch.save(labels_vec,   LBL_PATH)
    print("Saved to", SAVE_DIR)
    print("Features shape:", feats_matrix.shape, " Labels shape:", labels_vec.shape)


In [ ]:
print(torch.cuda.is_available())
print(next(base.parameters()).device)
